# Hyperparameter tuning XGBOOST (readmission as a reference)

## 0. Package loading and installation

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
import time

#conda install -c conda-forge \
#    numpy \
#    scipy \
#    pandas \
#    pyarrow \
#    scikit-survival \
#    spyder \
#    lifelines

# conda install -c conda-forge fastparquet
# conda install -c conda-forge xgboost
# conda install -c conda-forge pytorch cpuonly
# conda install -c pytorch pytorch cpuonly
# conda install -c conda-forge matplotlib
# conda install -c conda-forge seaborn
# conda install spyder-notebook -c spyder-ide
# conda install notebook nbformat nbconvert
# conda install -c conda-forge xlsxwriter
# conda install -c conda-forge shap

# import subprocess, sys

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "matplotlib"
# ])

# subprocess.check_call([
#     sys.executable,
#     "-m",
#     "pip",
#     "install",
#     "seaborn"
# ])

print("numpy:", np.__version__)


from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

#Dput
def dput_df(df, digits=6):
    data = {
        "columns": list(df.columns),
        "data": [
            [round(x, digits) if isinstance(x, (float, np.floating)) else x
             for x in row]
            for row in df.to_numpy()
        ]
    }
    print(data)


#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")
#Tabyl function
def tabyl(series):
    counts = series.value_counts(dropna=False)
    props = series.value_counts(normalize=True, dropna=False)
    return pd.DataFrame({"value": counts.index,
                         "n": counts.values,
                         "percent": props.values})
#clean_names
import re

def clean_names(df):
    """
    Mimic janitor::clean_names for pandas DataFrames.
    - Lowercase
    - Replace spaces and special chars with underscores
    - Remove non-alphanumeric/underscore
    """
    new_cols = []
    for col in df.columns:
        # lowercase
        col = col.lower()
        # replace spaces and special chars with underscore
        col = re.sub(r"[^\w]+", "_", col)
        # strip leading/trailing underscores
        col = col.strip("_")
        new_cols.append(col)
    df.columns = new_cols
    return df

numpy: 2.0.1


## Load data

In [ ]:

from pathlib import Path

BASE_DIR = Path(
    r"G:\My Drive\Alvacast\SISTRAT 2023\data\20241015_out\pred1"
)


import pickle

with open(BASE_DIR / "imputations_list_jan26.pkl", "rb") as f:
    imputations_list_jan26 = pickle.load(f)


imputation_nodum_1 = pd.read_parquet(
    BASE_DIR / "imputation_nondum_1.parquet",
    engine="fastparquet"
)

X_reduced_imp0 = pd.read_parquet(
    BASE_DIR / "X_reduced_imp0.parquet",
    engine="fastparquet"
)

imputation_1 = pd.read_parquet(
    BASE_DIR / "imputation_1.parquet",
    engine="fastparquet"
)


# Quick check
glimpse(imputation_nodum_1)
glimpse(imputation_1)
glimpse(X_reduced_imp0)

Rows: 88504 | Columns: 41
readmit_time_from_adm_m        float64         84.93548387096774, 12.833333333333334, 13.733333333333333, 11.966666666666667, 1...
death_time_from_adm_m          float64         84.93548387096774, 87.16129032258064, 117.2258064516129, 98.93548387096774, 37.9...
adm_age_rec3                   float64         31.53, 20.61, 42.52, 60.61, 45.08
porc_pobr                      float64         0.175679117441177, 0.187835901975632, 0.130412444472313, 0.133759185671806, 0.08...
dit_m                          float64         15.967741935483872, 5.833333333333334, 0.4752688172043005, 6.966666666666667, 6....
sex_rec                        object          man, man, man, woman, man
tenure_status_household        object          stays temporarily with a relative, owner/transferred dwellings/pays dividends, s...
cohabitation                   object          alone, family of origin, with couple/children, with couple/children, family of o...
sub_dep_icd10_status           obj

In [4]:
if isinstance(imputations_list_jan26, list) and len(imputations_list_jan26) > 0:
    print("First element type:", type(imputations_list_jan26[0]))
    if isinstance(imputations_list_jan26[0], dict):
        print("First element keys:", imputations_list_jan26[0].keys())
    elif isinstance(imputations_list_jan26[0], (pd.DataFrame, np.ndarray)):
        print("First element shape:", imputations_list_jan26[0].shape)


First element type: <class 'pandas.DataFrame'>
First element shape: (88504, 56)


This code block:

1.  **Imports the `pickle` library**: This library implements binary protocols for serializing and de-serializing a Python object structure.
2.  **Specifies the `file_path`**: It points to the `.pkl` file you selected.
3.  **Opens the file in binary read mode (`'rb'`)**: This is necessary for loading pickle files.
4.  **Loads the object**: `pickle.load(f)` reads the serialized object from the file and reconstructs it in memory.
5.  **Prints confirmation and basic information**: It verifies that the file was loaded and shows the type of the loaded object, and some details about the first element if it's a list containing common data structures.

#### Compare databases (transformed and original)

Inspect and compare the column names of two datasets: the first imputation from imputations_list_jan26 (which likely contains dummy variables) and imputation_nodum_1 (which, as its name suggests, probably doesn't have dummy variables).


In [5]:
# Inspect columns of the first imputation
cols_first_imp = imputations_list_jan26[0].columns.tolist()
print("First imputation columns:", cols_first_imp[:10], "... total:", len(cols_first_imp))

# Inspect columns of imputation_no_dum
cols_nodum = imputation_nodum_1.columns.tolist()
print("No-dum columns:", cols_nodum[:10], "... total:", len(cols_nodum))

# Compare overlap
common_cols = set(cols_first_imp).intersection(cols_nodum)
missing_in_imp = [c for c in cols_nodum if c not in cols_first_imp]
missing_in_nodum = [c for c in cols_first_imp if c not in cols_nodum]

print("Common columns:", len(common_cols))
print("Missing in imputations_list_jan26:", missing_in_imp)

First imputation columns: ['adm_age_rec3', 'porc_pobr', 'dit_m', 'tenure_status_household', 'prim_sub_freq_rec', 'national_foreign', 'urbanicity_cat', 'ed_attainment_corr', 'evaluacindelprocesoteraputico', 'eva_consumo'] ... total: 56
No-dum columns: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'adm_age_rec3', 'porc_pobr', 'dit_m', 'sex_rec', 'tenure_status_household', 'cohabitation', 'sub_dep_icd10_status', 'any_violence'] ... total: 41
Common columns: 24
Missing in imputations_list_jan26: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'sex_rec', 'cohabitation', 'sub_dep_icd10_status', 'any_violence', 'tr_outcome', 'adm_motive', 'first_sub_used', 'primary_sub_mod', 'tipo_de_vivienda_rec2', 'plan_type_corr', 'occupation_condition_corr24', 'marital_status_rec', 'readmit_event', 'death_event', 'center_id']


In [6]:
# Inspect columns of the first imputation
cols_first_imp_raw = imputation_1.columns.tolist()
print("First imputation columns:", cols_first_imp_raw[:10], "... total:", len(cols_first_imp_raw))

# Compare overlap
common_cols_raw = set(cols_first_imp_raw).intersection(cols_nodum)
missing_in_imp_raw = [c for c in cols_nodum if c not in cols_first_imp_raw]

print("Common columns:", len(common_cols_raw))
print("Missing in imputations_list_jan26:", missing_in_imp_raw)
print(common_cols_raw)

First imputation columns: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'adm_age_rec3', 'porc_pobr', 'dit_m', 'national_foreign', 'ethnicity', 'dg_psiq_cie_10_instudy', 'dg_psiq_cie_10_dg', 'dx_f3_mood'] ... total: 78
Common columns: 16
Missing in imputations_list_jan26: ['sex_rec', 'tenure_status_household', 'cohabitation', 'sub_dep_icd10_status', 'any_violence', 'prim_sub_freq_rec', 'tr_outcome', 'adm_motive', 'first_sub_used', 'primary_sub_mod', 'tipo_de_vivienda_rec2', 'plan_type_corr', 'occupation_condition_corr24', 'marital_status_rec', 'urbanicity_cat', 'ed_attainment_corr', 'evaluacindelprocesoteraputico', 'eva_consumo', 'eva_fam', 'eva_relinterp', 'eva_ocupacion', 'eva_sm', 'eva_fisica', 'eva_transgnorma', 'center_id']
{'dit_m', 'dx_f3_mood', 'death_event', 'national_foreign', 'readmit_time_from_adm_m', 'dg_psiq_cie_10_instudy', 'ethnicity', 'any_phys_dx', 'porc_pobr', 'dg_psiq_cie_10_dg', 'dx_f6_personality', 'adm_age_rec3', 'death_time_from_adm_m', 'polysubstance_stri

In [7]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars = ["adm_age_rec3", "porc_pobr", "dit_m"]

# Take one imputation (first element of the list) and merge with the no-dum dataset
df_imp = imputations_list_jan26[0]
df_nodum = imputation_nodum_1

merged_check = pd.merge(
    df_imp[key_vars],
    df_nodum[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check.shape[0]}")
print("Preview of merged check:")
print(merged_check.head())

#drop merge
del merged_check

Merged rows: 88516
Preview of merged check:
   adm_age_rec3  porc_pobr      dit_m
0         31.53   0.175679  15.967742
1         20.61   0.187836   5.833333
2         42.52   0.130412   0.475269
3         60.61   0.133759   6.966667
4         45.08   0.083189   6.903226


In [8]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars_raw = ['dit_m',
            'readmit_time_from_adm_m',
            'death_time_from_adm_m',
            'adm_age_rec3']
# Take one imputation (first element of the list) and merge with the no-dum dataset
df_raw = imputation_1

merged_check_raw = pd.merge(
    df_imp[key_vars],
    df_raw[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check_raw.shape[0]}")
print("Preview of merged check:")
print(merged_check_raw.head())
print(f"{(merged_check_raw.shape[0] / imputation_1.shape[0] * 100):.2f}%")
#drop merge
del merged_check_raw

Merged rows: 88516
Preview of merged check:
   adm_age_rec3  porc_pobr      dit_m
0         31.53   0.175679  15.967742
1         20.61   0.187836   5.833333
2         42.52   0.130412   0.475269
3         60.61   0.133759   6.966667
4         45.08   0.083189   6.903226
100.01%


### Create bins for followup (landmarks)

This code prepares your data for survival analysis. It extracts the time until an event (like readmission or death) and whether that event actually happened for each patient from the df_nodum dataset. Then, it automatically creates a set of important time points, called an 'evaluation grid', which are specific moments to assess the model's performance on both readmission and death outcomes.


In [9]:
import numpy as np

# Required columns for survival outcomes
required = ["readmit_time_from_disch_m", "readmit_event",
            "death_time_from_disch_m", "death_event"]

# Check that df_raw has all required columns
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(f"df_nodum is missing columns: {missing}")

# Create time/event arrays directly from df_raw
time_readm = df_raw["readmit_time_from_adm_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_adm_m"].to_numpy()
event_death = (df_nodum["death_event"].to_numpy() == 1)

print("Arrays created for df_raw:")
print("Readmission times:", time_readm[:5])
print("Readmission events:", event_readm[:5])
print("Death times:", time_death[:5])
print("Death events:", event_death[:5])

# Build evaluation grids (quantiles of event times)
event_times_readm = time_readm[event_readm]
event_times_death = time_death[event_death]

if len(event_times_readm) < 5 or len(event_times_death) < 5:
    raise ValueError("Too few events in df_raw to build reliable time grids.")

times_eval_readm = np.unique(np.quantile(event_times_readm, np.linspace(0.05, 0.95, 50)))
times_eval_death = np.unique(np.quantile(event_times_death, np.linspace(0.05, 0.95, 50)))

print("Eval times (readmission):", times_eval_readm[:5], "...", times_eval_readm[-5:])
print("Eval times (death):", times_eval_death[:5], "...", times_eval_death[-5:])


Arrays created for df_raw:
Readmission times: [84.93548387 12.83333333 13.73333333 11.96666667 14.25806452]
Readmission events: [False  True  True  True  True]
Death times: [ 84.93548387  87.16129032 117.22580645  98.93548387  37.93548387]
Death events: [False False False False False]
Eval times (readmission): [3.93548387 4.77419355 5.45058701 6.06492649 6.67741935] ... [54.44173469 58.41566162 63.23333333 68.54767171 74.68983871]
Eval times (death): [4.16290323 5.43022383 6.68564845 8.24254115 9.77961817] ... [81.92700461 85.41186103 88.78518762 93.5538183  99.21935484]


 ## Prepare data


First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [10]:
import numpy as np

# Step 1. Extract survival outcomes directly from df_raw
time_readm = df_raw["readmit_time_from_adm_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_adm_m"].to_numpy()
event_death = (df_raw["death_event"].to_numpy() == 1)

# Step 2. Build structured arrays (Surv objects)
y_surv_readm = np.empty(len(time_readm), dtype=[("event", "?"), ("time", "<f8")])
y_surv_readm["event"] = event_readm
y_surv_readm["time"] = time_readm

y_surv_death = np.empty(len(time_death), dtype=[("event", "?"), ("time", "<f8")])
y_surv_death["event"] = event_death
y_surv_death["time"] = time_death

# Step 3. Replicate across imputations
n_imputations = len(imputations_list_jan26)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

import numpy as np

def correct_competing_risks(X_list, y_readm_list, y_death_list):
    """
    Adjust survival outcomes for competing risks (death vs. readmission).

    Parameters
    ----------
    X_list : list of pd.DataFrame
        Imputed predictor datasets (same rows across imputations).
    y_readm_list : list of structured arrays
        Surv(event, time) arrays for readmission.
    y_death_list : list of structured arrays
        Surv(event, time) arrays for death.

    Returns
    -------
    y_readm_corrected_list : list of structured arrays
        Corrected readmission outcomes (death treated as censoring).
    """
    corrected = []
    for y_readm, y_death in zip(y_readm_list, y_death_list):
        y_corr = y_readm.copy()
        # If patient died before readmission → censor at death time
        for i in range(len(y_corr)):
            if y_death["event"][i] and y_death["time"][i] < y_corr["time"][i]:
                y_corr["event"][i] = False
                y_corr["time"][i] = y_death["time"][i]
        corrected.append(y_corr)
    return corrected


# Step 4. Apply correction
y_surv_readm_list_corrected = correct_competing_risks(
    imputations_list_jan26,
    y_surv_readm_list,
    y_surv_death_list
)

In [11]:
# Check type and length
type(y_surv_readm_list_corrected), len(y_surv_readm_list_corrected)

# Look at the first element
y_surv_readm_list_corrected[0][:5]   # first 5 rows


array([(False, 84.93548387), ( True, 12.83333333), ( True, 13.73333333),
       ( True, 11.96666667), ( True, 14.25806452)],
      dtype=[('event', '?'), ('time', '<f8')])

In [12]:
glimpse(imputations_list_jan26[0])
print(y_surv_readm_list_corrected[0].shape, y_surv_readm_list_corrected[0].dtype)

Rows: 88504 | Columns: 56
adm_age_rec3                   float64         31.53, 20.61, 42.52, 60.61, 45.08
porc_pobr                      float64         0.175679117441177, 0.187835901975632, 0.130412444472313, 0.133759185671806, 0.08...
dit_m                          float64         15.967741935483872, 5.833333333333334, 0.4752688172043005, 6.966666666666667, 6....
tenure_status_household        int64           3, 0, 3, 0, 3
prim_sub_freq_rec              int64           1, 2, 2, 2, 2
national_foreign               int32           0, 0, 0, 0, 0
urbanicity_cat                 int64           0, 0, 0, 0, 0
ed_attainment_corr             float64         1.0, 2.0, 1.0, 1.0, 2.0
evaluacindelprocesoteraputico  int64           0, 2, 2, 2, 0
eva_consumo                    int64           0, 2, 2, 1, 0
eva_fam                        int64           1, 2, 2, 1, 0
eva_relinterp                  int64           0, 2, 2, 1, 0
eva_ocupacion                  int64           0, 2, 2, 2, 1
eva_sm     

# ML

### Advanced Survival Modeling: XGBoost & Stratified Evaluation

In this section, we transition to a Gradient Boosted Decision Tree (GBDT) framework using XGBoost. This approach serves as a robust, non-linear benchmark to validate findings from the neural network, specifically optimized for high-imbalance survival data (approx. 4% death rate).

#### Methodological Framework:
* **Cox-Objective Boosting:** We utilize the `survival:cox` objective, which optimizes the Cox partial log-likelihood within a boosting architecture. This allows the model to learn complex non-linear risk functions and interactions without assuming proportional hazards or requiring manual interaction terms.

* **Stratified 5-Fold Cross-Validation:** To ensure robustness across diverse treatment modalities, we implement `StratifiedKFold` based on Plan Type (Outpatient, Intensive, Residential). This guarantees that every validation fold maintains the same distribution of clinical settings as the full dataset, preventing the model from overfitting to the majority treatment type.

* **Robust Metrics (IPCW & IBS):** Instead of standard AUC, we optimize for **Uno's C-Index (Inverse Probability of Censoring Weighting)**. This metric is statistically consistent for censored data and prevents bias when evaluating long-term outcomes in unbalanced datasets. We additionally calculate the **Integrated Brier Score (IBS)** to assess the calibration of the predicted survival probabilities.

#### Hyperparameter Optimization:
Given the extreme class imbalance, we employ a **Stratified Randomized Search** over a dense parameter grid. This process tunes critical regularization parameters (`min_child_weight`, `gamma`, `reg_alpha`) to prevent overfitting to the majority class (survivors) while maximizing discrimination on the minority class (events).

#### Breslow Estimation:
To bridge the gap between XGBoost's raw risk scores (log-hazards) and interpretable probabilities needed for calibration metrics, we explicitly compute the **Breslow Estimator**. This reconstructs the baseline survival function S0(t), allowing us to project absolute survival probabilities S(t|x) for any patient at any time point.

In [13]:
# ⚡ XGBoost Tuning (Peer-Review Standard: 5-Fold Stratified, CPU-only)

import numpy as np
import pandas as pd
import xgboost as xgb
import os

from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sksurv.metrics import concordance_index_ipcw, concordance_index_censored

import time
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# =========================
# 1. SETUP & DATA
# =========================
print("⚡ Preparing data for Robust XGBoost Tuning (CPU-only)...")

total_start_time = time.time()

try:
    if 'imputations_list_jan26' in locals():
        df_tune = imputations_list_jan26[0].copy()
        y_tune_struct = y_surv_readm_list_corrected[0]
    elif 'imputations_list' in locals():
        df_tune = imputations_list[0].copy()
        y_tune_struct = y_surv_readm_list_corrected[0]
    else:
        df_tune = X_train.copy()
        y_tune_struct = y_surv_readm_list_corrected[0]

    print(f"   Data Shape: {df_tune.shape}")
    print(f"   Target: Readmission (Events: {y_tune_struct['event'].sum()})")

except Exception as e:
    raise ValueError(f"❌ Data Error: {e}. Please run data loading steps first.")

# =========================
# 2. STRATIFICATION
# =========================
def get_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns:
        labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns:
        labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns:
        labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns:
        labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

strat_labels = get_stratification_labels(df_tune)

# XGBoost survival encoding
y_xgb_label = np.where(
    y_tune_struct['event'],
    y_tune_struct['time'],
    -y_tune_struct['time']
)

# =========================
# 3. SEARCH SPACE
# =========================
param_grid = {
    'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 5, 10, 20, 50],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 1, 5, 10],
    'reg_lambda': [0.1, 1, 5, 10, 20],
    'gamma': [0, 0.1, 0.5, 1, 2]
}

N_ITER = 50
param_list = list(ParameterSampler(param_grid, n_iter=N_ITER, random_state=42))

# =========================
# 4. TUNING LOOP
# =========================
print(f"\n🚀 Starting Exhaustive Search ({N_ITER} combos)")
print("   Strategy: 5-Fold Stratified CV")
print("   Metric: Uno's C-Index (IPCW)")
N_CORES = max(1, os.cpu_count() - 3)
print(f"   Backend: CPU-only | Threads: {N_CORES} (leaving 3 free)")

results = []
start_time = time.time()

for i, params in enumerate(param_list, 1):

    # Fixed parameters
    params.update({
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'nthread': N_CORES,          # 👈 CPU parallelism
        'verbosity': 0
    })

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in skf.split(df_tune, strat_labels):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1500,
            evals=[(dval, 'val')],
            early_stopping_rounds=30,
            verbose_eval=False
        )

        risk_scores = model.predict(dval)

        try:
            c_val = concordance_index_ipcw(
                y_tr_struct, y_va_struct, risk_scores
            )[0]
        except Exception:
            c_val = concordance_index_censored(
                y_va_struct['event'],
                y_va_struct['time'],
                risk_scores
            )[0]

        fold_scores.append(c_val)

    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    results.append({**params, 'Unos_C_Index': avg_score, 'Std_Dev': std_score})

    if i % 5 == 0:
        elapsed = (time.time() - start_time) / 60
        best_so_far = max(r['Unos_C_Index'] for r in results)
        print(
            f"   [{i}/{N_ITER}] "
            f"Best: {best_so_far:.4f} | "
            f"Current: {avg_score:.4f} (±{std_score:.3f}) | "
            f"{elapsed:.1f} min"
        )

# =========================
# 5. EXPORT RESULTS
# =========================
total_duration_min = (time.time() - total_start_time) / 60
print(f"\n🏁 Total Execution Time: {total_duration_min:.2f} minutes")

df_results = (
    pd.DataFrame(results)
      .sort_values(by='Unos_C_Index', ascending=False)
)

best_config = df_results.iloc[0].to_dict()

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"XGB_Robust_Tuning_5Fold_CPU_{timestamp_str}.csv"
df_results.to_csv(filename, index=False)

print("\n🏆 Tuning Complete!")
print(f"   Best C-Index: {best_config['Unos_C_Index']:.4f} "
      f"(±{best_config['Std_Dev']:.4f})")
print(f"   Optimal Config: {best_config}")
print(f"💾 Saved to: {filename}")

⚡ Preparing data for Robust XGBoost Tuning (CPU-only)...
   Data Shape: (88504, 56)
   Target: Readmission (Events: 19070)

🚀 Starting Exhaustive Search (50 combos)
   Strategy: 5-Fold Stratified CV
   Metric: Uno's C-Index (IPCW)
   Backend: CPU-only | Threads: 29 (leaving 3 free)
   [5/50] Best: 0.6225 | Current: 0.6219 (±0.005) | 1.3 min
   [10/50] Best: 0.6225 | Current: 0.6183 (±0.005) | 2.8 min
   [15/50] Best: 0.6227 | Current: 0.6164 (±0.004) | 4.9 min
   [20/50] Best: 0.6228 | Current: 0.6223 (±0.006) | 7.6 min
   [25/50] Best: 0.6228 | Current: 0.6167 (±0.006) | 9.7 min
   [30/50] Best: 0.6228 | Current: 0.6225 (±0.005) | 11.8 min
   [35/50] Best: 0.6228 | Current: 0.6222 (±0.005) | 13.1 min
   [40/50] Best: 0.6230 | Current: 0.6158 (±0.006) | 14.5 min
   [45/50] Best: 0.6230 | Current: 0.6225 (±0.005) | 16.6 min
   [50/50] Best: 0.6231 | Current: 0.6226 (±0.006) | 18.4 min

🏁 Total Execution Time: 18.44 minutes

🏆 Tuning Complete!
   Best C-Index: 0.6231 (±0.0053)
   Optimal

In [16]:
#@title ⚡ XGBoost Robust Tuning (Time Tracked & Memory Safe)
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sksurv.metrics import concordance_index_ipcw, concordance_index_censored

import os
import gc
import time
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# =========================
# CPU CONFIGURATION
# =========================
N_CORES = max(1, os.cpu_count() - 3)
print(f"⚙️ Using {N_CORES} CPU cores (leaving 3 free)")

total_start_time = time.time()

# =========================
# STRATIFICATION HELPER
# =========================
def get_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns:
        labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns:
        labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns:
        labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns:
        labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

strat_labels = get_stratification_labels(df_tune)

# XGBoost survival encoding
y_xgb_label = np.where(
    y_tune_struct['event'],
    y_tune_struct['time'],
    -y_tune_struct['time']
)

# =========================
# SEARCH SPACE
# =========================
param_grid = {
    'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 5, 10, 20, 50],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 1, 5, 10],
    'reg_lambda': [0.1, 1, 5, 10, 20],
    'gamma': [0, 0.1, 0.5, 1, 2]
}

N_ITER = 50
param_list = list(ParameterSampler(param_grid, n_iter=N_ITER, random_state=42))

# =========================
# TUNING LOOP
# =========================
print(f"\n🚀 Starting Exhaustive Search ({N_ITER} combos)")
print("   Strategy: 5-Fold Stratified CV")
print("   Metric: Uno's C-Index (IPCW)")
print("   Backend: CPU-only")

results = []

for i, params in enumerate(param_list, 1):

    params.update({
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'nthread': N_CORES,
        'verbosity': 0
    })

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    fold_scores = []

    for train_idx, val_idx in skf.split(df_tune, strat_labels):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1500,
            evals=[(dval, 'val')],
            early_stopping_rounds=30,
            verbose_eval=False
        )

        risk_scores = model.predict(dval)

        try:
            c_val = concordance_index_ipcw(
                y_tr_struct, y_va_struct, risk_scores
            )[0]
        except Exception:
            c_val = concordance_index_censored(
                y_va_struct['event'],
                y_va_struct['time'],
                risk_scores
            )[0]

        fold_scores.append(c_val)

        # Memory hygiene
        del model, dtrain, dval, risk_scores
        gc.collect()

    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    results.append({**params,
                    'Unos_C_Index': avg_score,
                    'Std_Dev': std_score})

    if i % 5 == 0:
        elapsed_min = (time.time() - total_start_time) / 60
        best_so_far = max(r['Unos_C_Index'] for r in results)
        print(
            f"   [{i}/{N_ITER}] "
            f"Best: {best_so_far:.4f} | "
            f"Current: {avg_score:.4f} | "
            f"Elapsed: {elapsed_min:.2f} min"
        )

# =========================
# EXPORT RESULTS
# =========================
total_duration_min = (time.time() - total_start_time) / 60
print(f"\n🏁 Total Execution Time: {total_duration_min:.2f} minutes")

df_results = (
    pd.DataFrame(results)
      .sort_values(by='Unos_C_Index', ascending=False)
)

best_config = df_results.iloc[0].to_dict()

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"XGB_Robust_Tuning_5Fold_CPU2_{timestamp_str}.csv"
df_results.to_csv(filename, index=False)

print("\n🏆 Tuning Complete!")
print(f"   Best C-Index: {best_config['Unos_C_Index']:.4f}")
print(f"   Std Dev: {best_config['Std_Dev']:.4f}")
print(f"💾 Saved to: {filename}")

⚙️ Using 29 CPU cores (leaving 3 free)

🚀 Starting Exhaustive Search (50 combos)
   Strategy: 5-Fold Stratified CV
   Metric: Uno's C-Index (IPCW)
   Backend: CPU-only
   [5/50] Best: 0.6224 | Current: 0.6219 | Elapsed: 1.56 min
   [10/50] Best: 0.6224 | Current: 0.6189 | Elapsed: 3.24 min
   [15/50] Best: 0.6229 | Current: 0.6167 | Elapsed: 5.49 min
   [20/50] Best: 0.6230 | Current: 0.6230 | Elapsed: 8.20 min
   [25/50] Best: 0.6230 | Current: 0.6190 | Elapsed: 10.05 min
   [30/50] Best: 0.6230 | Current: 0.6219 | Elapsed: 12.10 min
   [35/50] Best: 0.6230 | Current: 0.6222 | Elapsed: 13.43 min
   [40/50] Best: 0.6232 | Current: 0.6143 | Elapsed: 14.75 min
   [45/50] Best: 0.6232 | Current: 0.6221 | Elapsed: 16.68 min
   [50/50] Best: 0.6232 | Current: 0.6228 | Elapsed: 18.37 min

🏁 Total Execution Time: 18.37 minutes

🏆 Tuning Complete!
   Best C-Index: 0.6232
   Std Dev: 0.0060
💾 Saved to: XGB_Robust_Tuning_5Fold_CPU2_20260204_2319.csv


In [21]:
print(best_config)

{'subsample': 0.8, 'reg_lambda': 5.0, 'reg_alpha': 0.1, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.02, 'gamma': 0.5, 'colsample_bytree': 0.7, 'objective': 'survival:cox', 'eval_metric': 'cox-nloglik', 'tree_method': 'hist', 'nthread': 29, 'verbosity': 0, 'Unos_C_Index': 0.6232038890484929, 'Std_Dev': 0.0060480692974189875}


In [1]:
#@title 🏆 Optimal XGBoost Configuration (Interactive Summary)
import pandas as pd

# 1. Define the Optimal Configuration Data (from your tuning results)
data = [
    {"Category":"Boosting","Parameter":"learning_rate","Value":0.02,
     "Description":"Moderate learning rate; balances convergence speed and generalization."},

    {"Category":"Tree Structure","Parameter":"max_depth","Value":6,
     "Description":"Moderately deep trees; captures non-linear interactions without excessive overfitting."},

    {"Category":"Imbalance Handling","Parameter":"min_child_weight","Value":10,
     "Description":"High threshold; prevents leaf nodes from isolating rare readmission events (noise control)."},

    {"Category":"Regularization","Parameter":"reg_alpha (L1)","Value":0.1,
     "Description":"Mild L1 penalty; allows feature retention while limiting over-sparsification."},

    {"Category":"Regularization","Parameter":"reg_lambda (L2)","Value":5.0,
     "Description":"Strong L2 penalty; stabilizes coefficients and mitigates multicollinearity."},

    {"Category":"Stochasticity","Parameter":"subsample","Value":0.8,
     "Description":"Uses 80% of rows per tree; improves robustness while preserving signal."},

    {"Category":"Stochasticity","Parameter":"colsample_bytree","Value":0.7,
     "Description":"Uses 70% of features per tree; encourages predictor diversity."},

    {"Category":"Tree Structure","Parameter":"gamma","Value":0.5,
     "Description":"Requires minimum loss reduction for splits; discourages weak partitions."},

    {"Category":"Model Specification","Parameter":"objective","Value":"survival:cox",
     "Description":"Cox proportional hazards objective for right-censored time-to-event modeling."},

    {"Category":"Optimization","Parameter":"eval_metric","Value":"cox-nloglik",
     "Description":"Negative partial log-likelihood aligned with Cox loss optimization."},

    {"Category":"Computation","Parameter":"tree_method","Value":"hist",
     "Description":"Histogram-based algorithm for efficient large-scale training."},

    {"Category":"Computation","Parameter":"CPU threads","Value":29,
     "Description":"Parallel execution using available CPU cores (system reserve applied)."},

    {"Category":"Performance","Parameter":"🏆 Uno's C-Index (IPCW)","Value":0.6232,
     "Description":"Moderate discriminative performance for readmission risk prediction."},

    {"Category":"Stability","Parameter":"Standard Deviation","Value":"±0.0060",
     "Description":"Low variability across 5-fold CV; stable tuning solution."}
]

pd.set_option('display.max_colwidth', None)
# 2. Create DataFrame
df_optimal_config = pd.DataFrame(data)

df_optimal_config

,Category,Parameter,Value,Description
0,Boosting,learning_rate,0.02,Moderate learning rate; balances convergence speed and generalization.
1,Tree Structure,max_depth,6,Moderately deep trees; captures non-linear interactions without excessive overfitting.
2,Imbalance Handling,min_child_weight,10,High threshold; prevents leaf nodes from isolating rare readmission events (noise control).
3,Regularization,reg_alpha (L1),0.1,Mild L1 penalty; allows feature retention while limiting over-sparsification.
4,Regularization,reg_lambda (L2),5.0,Strong L2 penalty; stabilizes coefficients and mitigates multicollinearity.
5,Stochasticity,subsample,0.8,Uses 80% of rows per tree; improves robustness while preserving signal.
6,Stochasticity,colsample_bytree,0.7,Uses 70% of features per tree; encourages predictor diversity.
7,Tree Structure,gamma,0.5,Requires minimum loss reduction for splits; discourages weak partitions.
8,Model Specification,objective,survival:cox,Cox proportional hazards objective for right-censored time-to-event modeling.
9,Optimization,eval_metric,cox-nloglik,Negative partial log-likelihood aligned with Cox loss optimization.


Hyperparameter tuning identified an optimal configuration with a moderate learning rate (0.02) and tree depth of six, indicating a balance between convergence efficiency and model complexity. Strong L2 regularization (λ = 5.0), along with row and feature subsampling, constrained overfitting while preserving predictive signal. This combination enabled the model to capture non-linear interactions while maintaining stable generalization performance (Uno’s C-index = 0.623 ± 0.006).

In [22]:
#@title ⚡ XGBoost "Deep & Slow" Deterministic Strategy
import numpy as np
import pandas as pd
import xgboost as xgb
import os
import gc
import time
from datetime import datetime
import warnings

# NOTE: Using ParameterGrid for deterministic search instead of ParameterSampler
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sksurv.metrics import concordance_index_ipcw, concordance_index_censored

warnings.filterwarnings("ignore")

# =========================
# 1. SETUP & DATA
# =========================
# Assumes df_tune and y_tune_struct are already loaded in your environment
print("⚡ Preparing data for Deep & Slow Strategy Check...")

total_start_time = time.time()

# CPU Configuration
N_CORES = max(1, os.cpu_count() - 3)
print(f"⚙️ Using {N_CORES} CPU cores (leaving 3 free)")

# Stratification Helper
def get_stratification_labels(df):
    labels = np.zeros(len(df), dtype=int)
    if 'plan_type_corr_pg_pr' in df.columns:
        labels[df['plan_type_corr_pg_pr'] == 1] = 1
    if 'plan_type_corr_m_pr' in df.columns:
        labels[df['plan_type_corr_m_pr'] == 1] = 2
    if 'plan_type_corr_pg_pai' in df.columns:
        labels[df['plan_type_corr_pg_pai'] == 1] = 3
    if 'plan_type_corr_m_pai' in df.columns:
        labels[df['plan_type_corr_m_pai'] == 1] = 4
    return labels

try:
    strat_labels = get_stratification_labels(df_tune)
    
    # XGBoost survival encoding
    y_xgb_label = np.where(
        y_tune_struct['event'],
        y_tune_struct['time'],
        -y_tune_struct['time']
    )
    
    print(f"   Data Shape: {df_tune.shape}")
    print(f"   Target: Readmission (Events: {y_tune_struct['event'].sum()})")

except NameError:
    raise ValueError("❌ 'df_tune' or 'y_tune_struct' not found. Please run data loading steps first.")

# =========================
# 2. DETERMINISTIC SEARCH SPACE
# =========================
# We fix the "noise" parameters to their known robust values
# and only vary the Deep & Slow drivers.
param_grid = {
    # The Core "Deep & Slow" Hypothesis
    'learning_rate': [0.001, 0.005],
    'max_depth': [8, 10],
    'reg_lambda': [5, 10],
    
    # Fixed Robust Stabilizers (from previous analysis)
    'min_child_weight': [15], 
    'subsample': [0.8],
    'colsample_bytree': [0.7],
    'reg_alpha': [0.5],
    'gamma': [0.5]
}

# Generate the exact list of combinations (2 x 2 x 2 = 8 runs)
param_list = list(ParameterGrid(param_grid))
N_ITER = len(param_list)

# =========================
# 3. TUNING LOOP
# =========================
print(f"\n🚀 Starting Deterministic Grid Search ({N_ITER} combos)")
print("   Strategy: Deep Trees + Low Learning Rate")
print("   Metric: Uno's C-Index (IPCW)")

results = []

for i, params in enumerate(param_list, 1):
    
    # Fixed parameters for Survival
    params.update({
        'objective': 'survival:cox',
        'eval_metric': 'cox-nloglik',
        'tree_method': 'hist',
        'nthread': N_CORES,
        'verbosity': 0
    })
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2125)
    fold_scores = []
    
    # Measure time per iteration to estimate remaining time
    iter_start = time.time()
    
    for train_idx, val_idx in skf.split(df_tune, strat_labels):
        X_tr, X_va = df_tune.iloc[train_idx], df_tune.iloc[val_idx]
        y_tr_xgb, y_va_xgb = y_xgb_label[train_idx], y_xgb_label[val_idx]
        y_tr_struct, y_va_struct = y_tune_struct[train_idx], y_tune_struct[val_idx]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr_xgb)
        dval = xgb.DMatrix(X_va, label=y_va_xgb)
        
        model = xgb.train(
            params,
            dtrain,
            # Increased rounds significantly for low learning rate
            num_boost_round=5000,
            evals=[(dval, 'val')],
            # Increased patience
            early_stopping_rounds=50,
            verbose_eval=False
        )
        
        risk_scores = model.predict(dval)
        
        try:
            c_val = concordance_index_ipcw(
                y_tr_struct, y_va_struct, risk_scores
            )[0]
        except Exception:
            c_val = concordance_index_censored(
                y_va_struct['event'],
                y_va_struct['time'],
                risk_scores
            )[0]
        
        fold_scores.append(c_val)
        
        # Memory hygiene
        del model, dtrain, dval, risk_scores
        gc.collect()
        
    avg_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    
    iter_duration = (time.time() - iter_start) / 60
    
    results.append({**params, 
                    'Unos_C_Index': avg_score, 
                    'Std_Dev': std_score})
    
    best_so_far = max(r['Unos_C_Index'] for r in results)
    print(
        f"   [{i}/{N_ITER}] "
        f"Best: {best_so_far:.4f} | "
        f"Current: {avg_score:.4f} (±{std_score:.4f}) | "
        f"Time: {iter_duration:.1f} min | "
        f"Params: LR={params['learning_rate']}, D={params['max_depth']}, L2={params['reg_lambda']}"
    )

# =========================
# 4. EXPORT RESULTS
# =========================
total_duration_min = (time.time() - total_start_time) / 60
print(f"\n🏁 Total Execution Time: {total_duration_min:.2f} minutes")

df_results = (
    pd.DataFrame(results)
      .sort_values(by='Unos_C_Index', ascending=False)
)

best_config = df_results.iloc[0].to_dict()

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"XGB_DeepSlow_Strategy_{timestamp_str}.csv"
df_results.to_csv(filename, index=False)

print("\n🏆 Deep & Slow Check Complete!")
print(f"   Best C-Index: {best_config['Unos_C_Index']:.4f}")
print(f"   Std Dev: {best_config['Std_Dev']:.4f}")
print(f"💾 Saved to: {filename}")

# 🏁 Total Execution Time: 13.24 minutes

# 🏆 Deep & Slow Check Complete!
#    Best C-Index: 0.6230
#    Std Dev: 0.0057
# 💾 Saved to: XGB_DeepSlow_Strategy_20260204_1826.csv

#The best "Deep & Slow" model achieved 0.6230, which is statistically identical 
#to (and technically slightly lower than) the 0.6232 from your random search.

⚡ Preparing data for Deep & Slow Strategy Check...
⚙️ Using 29 CPU cores (leaving 3 free)
   Data Shape: (88504, 56)
   Target: Readmission (Events: 19070)

🚀 Starting Deterministic Grid Search (8 combos)
   Strategy: Deep Trees + Low Learning Rate
   Metric: Uno's C-Index (IPCW)
   [1/8] Best: 0.6222 | Current: 0.6222 (±0.0058) | Time: 2.2 min | Params: LR=0.001, D=8, L2=5
   [2/8] Best: 0.6222 | Current: 0.6219 (±0.0058) | Time: 2.2 min | Params: LR=0.001, D=8, L2=10
   [3/8] Best: 0.6222 | Current: 0.6220 (±0.0055) | Time: 2.6 min | Params: LR=0.001, D=10, L2=5
   [4/8] Best: 0.6222 | Current: 0.6219 (±0.0055) | Time: 2.7 min | Params: LR=0.001, D=10, L2=10
   [5/8] Best: 0.6229 | Current: 0.6229 (±0.0061) | Time: 0.8 min | Params: LR=0.005, D=8, L2=5
   [6/8] Best: 0.6230 | Current: 0.6230 (±0.0057) | Time: 0.8 min | Params: LR=0.005, D=8, L2=10
   [7/8] Best: 0.6230 | Current: 0.6221 (±0.0058) | Time: 0.6 min | Params: LR=0.005, D=10, L2=5
   [8/8] Best: 0.6230 | Current: 0.6221 (